# E-commerce Demand Forecasting

**Weekly demand forecasts with a p90 stocking range and a promo planner · Python · Scikit-learn · Shiny**

## Hero Overview

A weekly demand forecaster for 320 products over 131 weeks, predicting 4 weeks and 3 months ahead with a p90 upper range, and cutting error 26% and 21% against the best standard baseline.

Filling stockout weeks in the history removes most of the under-forecasting, and a promo planner lets a product manager test a promotion before running it.

In [1]:
outcomes = {
    "4-Week Error (WAPE)": "15.2% vs 20.5% best baseline (\u221226%)",
    "3-Month Error (WAPE)": "18.9% vs 23.8% best baseline (\u221221%)",
    "Revenue-Weighted Error": "16.2% (4 wk) / 19.5% (3 mo)",
    "p90 Coverage of Total": "89.6% (4 wk) / 90.4% (3 mo)",
    "Stockout Fill": "17.6% \u2192 15.2% 4-week error",
    "Healthy Product Bias": "\u22123.9% (4 wk) / \u22127.2% (3 mo)",
    "Final-Decline Bias": "+42% (4 wk) / +72% (3 mo)",
    "Nov\u2013Dec Error": "13.4% (4 wk) / 20.3% (3 mo)"
}

pd.DataFrame(outcomes.items(), columns=["Metric", "Value"])

Metric,Value
4-Week Error (WAPE),15.2% vs 20.5% best baseline (−26%)
3-Month Error (WAPE),18.9% vs 23.8% best baseline (−21%)
Revenue-Weighted Error,16.2% (4 wk) / 19.5% (3 mo)
p90 Coverage of Total,89.6% (4 wk) / 90.4% (3 mo)
Stockout Fill,17.6% → 15.2% 4-week error
Healthy Product Bias,−3.9% (4 wk) / −7.2% (3 mo)
Final-Decline Bias,+42% (4 wk) / +72% (3 mo)
Nov–Dec Error,13.4% (4 wk) / 20.3% (3 mo)


In [2]:
# Radar (skill intensity) + Bubble cluster (implementation strength).
fig

Implementation Strength by System

In [3]:
# Technology Capability Network — concepts ↔ tools (bipartite, D3).

Demand Forecasting Network 
 Stockout recovery, leak-free features, gradient-boosted forecasting, and p90 planning for weekly product demand.

# Storyline

## Context

A product manager planning promotions and buys needs weekly demand per product, and a forecast that looks strong on average can hide a large error. Here the average hid it: products in final decline were over-forecast by 42% while healthy ones were under-forecast by 3.9%, and the two cancelled to a −1.3% overall bias. The data is a synthetic Kaggle set of 320 products, 8 categories and 5 regions over 131 weeks (24,310 rows), which records true demand including weeks with empty shelves, so forecasts can be scored against what customers actually wanted. See the [write-up](https://writing.karthikiyer.info/demand-forecast-bias/) and the [live project](https://projects.karthikiyer.info/#forecast).

## Workflow

Fill stockout weeks → build features as of the forecast date → train one gradient-boosted model across all products → train a p90 model → score with rolling-origin backtests against true demand → plan buys on the p90 of the horizon total → precompute promo-off and promo-on forecasts for the planner.

## Technical Implementation

- **Stockout recovery:** each stockout week is replaced by the average of the product's last two in-stock weeks, using earlier data only. Recorded sales capture only 14% of real demand in those weeks; filling them moved bias from −4.9% to −1.3%.
- **Features:** recent sales, trend, the usual seasonal pattern for the coming week, same week last year, product age, price, and whether a promotion is planned. Returns and two simulator answer-key columns are excluded. Scrambling every week after the forecast date and rebuilding inputs left all 3,380 tested rows unchanged.
- **Model:** one HistGradientBoosting model across all 320 products, because per-product ETS needs about two years of history that no product had. Predicting residuals from a seasonal baseline lifted the November rise from 1.22× to 1.40× (the real rise was 1.49×).
- **Baselines:** three spreadsheet rules and two ETS models; textbook ETS missed 22.0% at 4 weeks and 33.7% at 3 months, and same week last year missed 28.5% and 27.9%.
- **Stocking range:** a second model predicts the p90, and buys are planned on the p90 of the total, not the sum of weekly p90s, which covered 96–97% of totals instead of 90%.
- **Promo planner:** a Shiny for Python app over precomputed promo-off and promo-on forecasts.

## Outcomes

The forecast missed 15.2% of units at 4 weeks and 18.9% at 3 months, against 20.5% and 23.8% for the best baseline (26% and 21% less error). The p90 of the total covered 89.6% and 90.4% of outcomes and plans 2–8% less stock than summing weekly p90s. Limits: the data is synthetic, so the numbers need checking on real sales; products in final decline are still over-forecast (+42% and +72%) even with a slowing-sales warning; the 3-month forecast is weakest in November and December (20.3%); and it forecasts demand, not order size, with no stock on hand, lead times or minimum order quantities.